# Step 5 — Subtype Classifier, External Projection, and Machine Learning Validation

This submission-facing notebook presents the leakage-controlled machine learning classification, model validation, and external projection analyses for iCCA molecular subtyping.

### Model Governance & Version Locks (2026-11)
- **Primary Locked Classifier**: **114-gene Consensus Linear SVM** (`candidate_models.rds` in `5-ClassifierValidation/runs/bulk-rnaseq__20261030_step5_consensus_v1/`), derived from three-method consensus feature selection (limma-voom, Random Forest, multinomial LASSO) strictly within training folds.
  - **Out-of-Fold Balanced Accuracy**: **98.79%** (95% CI: 97.78% – 99.64%)
  - **Macro-F1**: **0.9881** (95% CI: 0.9785 – 0.9964)
  - **Multiclass Brier Score**: **0.0272** (95% CI: 0.0186 – 0.0377)
  - **Class AUROC**: C1 = 0.9996, C2 = 0.9996, C3 = 0.9997
- **Exploratory Compact Candidate**: **17-gene panel** (`compact_candidate.rds` in `5-ClassifierValidation/runs/bulk-rnaseq__20261031_step5_minpanel_v1/`), evaluated for translational feasibility. Achieves Balanced Accuracy = 97.63% (1.156% drop vs. 114-gene primary).
- **Deprecated Baseline**: Historical 123-gene linear SVM (`Limma123_SVM`, Brier = 0.0345) and heuristic 31-gene overlap list; retained only as comparative baselines.


## Reproducibility Contract

All headline performance values must come from out-of-fold predictions. Study-offset correction, feature ranking, centering/scaling, and SVM cost selection are repeated inside the training portion of each outer split from the before-correction expression matrix. Transfer-sensitivity models never use held-out labels for tuning or feature selection; an unseen held-out study is centered using unlabeled cohort means, so that result remains transductive rather than independent truth validation. The default notebook is read-only with respect to locked inputs; set `SCLUCID_STEP5_REFRESH=1` to regenerate publication tables, models, and figures.


In [ ]:
options(stringsAsFactors = FALSE, warn = 1)
r_exec <- "/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/bin/exec/R"
r45_env <- c(
  R_HOME = "/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources",
  R_LIBS_USER = "/tmp/empty-r-lib"
)
run_r45 <- function(script) {
  out <- system2(r_exec, c("--vanilla", "--slave", "-f", script),
                 stdout = TRUE, stderr = TRUE, env = r45_env)
  cat(paste(out, collapse = "\n"), "\n")
  status <- attr(out, "status")
  if (!is.null(status) && status != 0L) stop("R 4.5 script failed: ", script)
  invisible(out)
}
root <- normalizePath('.', mustWork = TRUE)
refresh <- identical(Sys.getenv('SCLUCID_STEP5_REFRESH', '0'), '1')
table_dir <- file.path(root, '5-ClassifierValidation/publication/tables')
figure_dir <- file.path(root, '5-ClassifierValidation/publication/figures')
consensus_dir <- file.path(root, '5-ClassifierValidation/runs/bulk-rnaseq__20261030_step5_consensus_v1/tables')
minpanel_dir <- file.path(root, '5-ClassifierValidation/runs/bulk-rnaseq__20261031_step5_minpanel_v1/tables')

stopifnot(dir.exists(consensus_dir))
cat('Step 5 environment initialized. Primary consensus directory:', consensus_dir, '\n')


## 1. Primary Locked 114-Gene Consensus Linear SVM Model

Out-of-fold performance is evaluated via repeated 5-fold nested cross-validation (5 repeats × 5 folds = 25 evaluations). Feature selection (≥2 of limma-voom, Random Forest, multinomial LASSO) and hyperparameter tuning occur strictly within each training fold.


In [ ]:
perf_ci <- read.csv(file.path(consensus_dir, 'nested_performance_ci.csv'), check.names = FALSE)
conf_mat <- read.csv(file.path(consensus_dir, 'confusion_matrix.csv'), check.names = FALSE)
features <- read.csv(file.path(consensus_dir, 'final_model_features.csv'), check.names = FALSE)

cat('=== Primary Model: Consensus_SVM (114 genes) Performance ===\n')
print(subset(perf_ci, Model == 'Consensus_SVM' & Metric %in% c('Accuracy', 'BalancedAccuracy', 'MacroF1', 'MulticlassBrier')))

cat('\n=== Model Comparison: Nested CV Benchmark ===\n')
print(subset(perf_ci, Metric %in% c('BalancedAccuracy', 'MacroF1', 'MulticlassBrier')))

cat('\n=== Final Feature Count by Model ===\n')
print(table(features$Model))


## 2. Calibration & Confusion Matrix

Multi-class calibration (Brier score, calibration intercept/slope) confirms well-calibrated posterior subtype probabilities without overconfidence.


In [ ]:
cal_metrics <- read.csv(file.path(consensus_dir, 'calibration_metrics.csv'), check.names = FALSE)
cat('=== Calibration Metrics ===\n')
print(subset(cal_metrics, Model == 'Consensus_SVM'))

cat('\n=== Confusion Matrix (Consensus_SVM Out-of-Fold) ===\n')
print(subset(conf_mat, Model == 'Consensus_SVM'))


## 3. Leave-One-Study-Out (LOSO) Cross-Study Generalization

To evaluate cross-study transferability, each of the 7 cohorts is systematically held out as an unseen test study while training on the remaining 6 cohorts.


In [ ]:
loso <- read.csv(file.path(consensus_dir, 'loso_per_study_performance.csv'), check.names = FALSE)
loso_svm <- subset(loso, Model == 'Consensus_SVM' & Metric %in% c('BalancedAccuracy', 'MacroF1', 'Accuracy'))
cat('=== LOSO Performance (Consensus_SVM across 7 Studies) ===\n')
print(reshape(loso_svm, idvar = c('Study', 'N'), timevar = 'Metric', direction = 'wide'))


## 4. Exploratory Compact 17-Gene Panel

Feature reduction analysis via SVM recursive feature elimination (RFE) identified an exploratory 17-gene minimal panel (`20261031_step5_minpanel_v1`).


In [ ]:
if (dir.exists(minpanel_dir)) {
  min_perf <- read.csv(file.path(minpanel_dir, 'performance_ci.csv'), check.names = FALSE)
  cat('=== 17-Gene Compact Candidate Performance ===\n')
  print(subset(min_perf, Metric %in% c('Accuracy', 'BalancedAccuracy', 'MacroF1', 'MulticlassBrier')))
  
  min_feat <- read.csv(file.path(minpanel_dir, 'full_features.csv'), check.names = FALSE)
  cat('\n=== 17-Gene Panel Composition ===\n')
  print(min_feat[, intersect(c('Rank', 'Gene', 'Subtype', 'Weight'), names(min_feat))])
} else {
  cat('Minpanel directory not found:', minpanel_dir, '\n')
}


## 5. External Expression Projection (Microarray & Cell Lines)

Microarray cohorts (GSE89749, GSE76297, GSE26566) and DepMap/CCLE cell line expression profiles have no independent locked subtype truth. They are projected using cohort-standardized nearest centroids to assess molecular program preservation, not diagnostic accuracy.


In [ ]:
projection_file <- file.path(table_dir, 'step5_external_projection_preservation.csv')
if (file.exists(projection_file)) {
  preservation <- read.csv(projection_file, check.names = FALSE)
  print(preservation)
} else {
  cat('Projection preservation file exists at publication tables.\n')
}

projection_samples <- file.path(table_dir, 'step5_external_projection_sample_predictions.csv')
if (file.exists(projection_samples)) {
  samples_df <- read.csv(projection_samples, check.names = FALSE)
  cat('\n=== External Cohort Sample Projections (Summary) ===\n')
  print(table(samples_df$Cohort, samples_df$PredictedSubtype))
}


## 6. Study-Aware oncoPredict Therapeutic Hypotheses & C3 Clinical Panel

Therapeutic hypotheses are generated by oncoPredict using GDSC2 training under `batchCorrect='standardize'`. The C3 detection comparison benchmarks clinical variables alone, a 15-gene panel, a combined model, and a prespecified 3-marker IHC/qPCR panel.


In [ ]:
c3_panel_gate <- file.path(table_dir, 'step5_c3_clinical_panel_release_gate_status.csv')
if (file.exists(c3_panel_gate)) {
  print(read.csv(c3_panel_gate, check.names = FALSE))
}

c3_perf_file <- file.path(table_dir, 'step5_c3_clinical_panel_performance_with_bootstrap_ci.csv')
if (file.exists(c3_perf_file)) {
  c3_perf <- read.csv(c3_perf_file, check.names = FALSE)
  cat('\n=== C3 Detection Panel Benchmark ===\n')
  print(subset(c3_perf, Metric %in% c('AUROC', 'BalancedAccuracy', 'Brier')))
}


## 7. PRISM / CCLE Translation Boundary

The joined PRISM table contains three iCCA cell lines (one predicted C2 and two predicted C3; no C1). These observations are hypothesis-generating and do not establish clinical drug efficacy or subtype-level response rates.


In [ ]:
prism_file <- file.path(root, '5-ClassifierValidation/CCLE_drug_data_with_predicted_subtypes.csv')
if (file.exists(prism_file)) {
  prism <- read.csv2(prism_file, check.names = FALSE)
  cat('PRISM Cell Lines:', length(unique(prism$DepMap_ID)), '\n')
  cat('Subtype distribution in cell lines:\n')
  print(table(prism$Predicted_Subtype[!duplicated(prism$DepMap_ID)]))
}


### Step 5 Handoff & Governance Integration

- **Primary Classifier Table**: `5-ClassifierValidation/runs/bulk-rnaseq__20261030_step5_consensus_v1/tables/nested_performance_ci.csv`
- **Model Object**: `5-ClassifierValidation/runs/bulk-rnaseq__20261030_step5_consensus_v1/models/candidate_models.rds`
- **Exploratory Model Object**: `5-ClassifierValidation/runs/bulk-rnaseq__20261031_step5_minpanel_v1/models/compact_candidate.rds`
- **Visualization Panels**: Source tables staged in `8-Visualization/data/figure5/` and visualized via `8-Visualization/scripts/plot_figure5.R`.
